<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/02_bigquery/05_ENARES_2024_STAGE2_competency_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05_ENARES_2024_STAGE2_competency_checks.ipynb

**Proyecto:** ENARES 2024 - Módulo CRS04  
**Stage:** 02  
**Objetivo:** verificar evidencias de competencias para el anexo integrado de Sprint y Competencias Stage 02.

Este notebook complementa los notebooks canónicos:

- `01_ENARES_2024_STAGE2_bigquery_setup.ipynb`
- `02_ENARES_2024_STAGE2_load_crs04_sav_to_bigquery.ipynb`
- `03_ENARES_2024_STAGE2_load_spss_metadata_to_bigquery.ipynb`
- `04_ENARES_2024_STAGE2_cloud_storage_report.ipynb`

## Issues cubiertos

- **Issue #16:** arquitectura por capas.
- **Issue #17:** carga raw y trazabilidad.
- **Issue #18:** metadata y gobernanza.
- **Extensión:** observabilidad, linaje y cierre parcial de competencias.

## Regla metodológica Stage 02

Este notebook **no hace merge**, **no recodifica**, **no crea indicadores**, **no modela** y **no interpreta resultados estadísticos**. Solo verifica evidencias ya producidas en Stage 02.

In [ ]:
# ============================================================
# 0. Instalación y autenticación
# ============================================================

!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow

from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
import pandas as pd
import os
import glob
import json
import hashlib

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = os.environ.get("PROJECT_ID") or input("Enter Google Cloud PROJECT_ID: ").strip()
if not PROJECT_ID:
    raise ValueError("PROJECT_ID no puede estar vacío.")

LOCATION = "US"

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"
DOCS_DIR = f"{ROOT_DRIVE}/docs"

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()

RAW_DATASET = "enares2024_crs04_raw"
EXPECTED_LAYER_DATASETS = [
    "enares2024_crs04_raw",
    "enares2024_crs04_cleaned",
    "enares2024_crs04_analytical",
]

RAW_TABLES = {
    "raw_crs04_cap100": {"source_file": "19_CRS04_CAP100.sav", "expected_rows": 18807, "expected_columns": 147},
    "raw_crs04_cap200": {"source_file": "20_CRS04_CAP200.sav", "expected_rows": 18807, "expected_columns": 523},
    "raw_crs04_cap248": {"source_file": "21_CRS04_CAP248.sav", "expected_rows": 18807, "expected_columns": 578},
    "raw_crs04_cap300": {"source_file": "22_CRS04_CAP300.sav", "expected_rows": 18807, "expected_columns": 51},
}

METADATA_TABLES = [
    "metadata_crs04_variables",
    "metadata_crs04_value_labels",
    "metadata_crs04_missing_codes",
    "metadata_crs04_source_files",
]

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print("PROJECT_ID:", PROJECT_ID)
print("LOCATION:", LOCATION)
print("LOG_DIR:", LOG_DIR)
print("DOCS_DIR:", DOCS_DIR)
print("RUN_UTC:", RUN_UTC)
print("Stage 02 competency checks initialized.")

## 1. Control metodológico de Stage 02

Este bloque deja evidencia explícita de acciones permitidas y prohibidas.

In [ ]:
STAGE = "STAGE02"

STAGE02_ALLOWED_ACTIONS = [
    "dataset_setup",
    "raw_ingestion",
    "metadata_preservation",
    "read_only_profiling",
    "technical_documentation",
    "security_documentation",
    "closure_reporting",
    "competency_evidence_checks",
]

STAGE02_FORBIDDEN_ACTIONS = [
    "merge",
    "recode",
    "derived_variables",
    "indicators",
    "statistical_models",
    "substantive_interpretation",
]

stage_policy = pd.DataFrame({
    "stage": [STAGE],
    "allowed_actions": ["; ".join(STAGE02_ALLOWED_ACTIONS)],
    "forbidden_actions": ["; ".join(STAGE02_FORBIDDEN_ACTIONS)],
    "checked_at_utc": [RUN_UTC],
})

stage_policy_output = f"{LOG_DIR}/ENARES_2024_STAGE2_competency_stage_policy.csv"
stage_policy.to_csv(stage_policy_output, index=False)

display(stage_policy)
print("Saved:", stage_policy_output)

## 2. Issue #16 — Verificación de arquitectura por capas

Verifica que existan los datasets:

- `enares2024_crs04_raw`
- `enares2024_crs04_cleaned`
- `enares2024_crs04_analytical`

In [ ]:
dataset_rows = []

for dataset_id in EXPECTED_LAYER_DATASETS:
    full_dataset_id = f"{PROJECT_ID}.{dataset_id}"
    try:
        dataset = client.get_dataset(full_dataset_id)
        exists = True
        location = dataset.location
        created = dataset.created.isoformat() if dataset.created else None
        modified = dataset.modified.isoformat() if dataset.modified else None
        error_message = None
    except Exception as exc:
        exists = False
        location = None
        created = None
        modified = None
        error_message = str(exc)

    dataset_rows.append({
        "project_id": PROJECT_ID,
        "dataset_id": dataset_id,
        "full_dataset_id": full_dataset_id,
        "exists": exists,
        "location": location,
        "created": created,
        "modified": modified,
        "competency": "Bases de Datos - arquitectura por capas",
        "checked_at_utc": RUN_UTC,
        "error_message": error_message,
    })

competency_dataset_check = pd.DataFrame(dataset_rows)

dataset_check_output = f"{LOG_DIR}/ENARES_2024_STAGE2_competency_dataset_layers_check.csv"
competency_dataset_check.to_csv(dataset_check_output, index=False)

display(competency_dataset_check)

if not competency_dataset_check["exists"].all():
    missing = competency_dataset_check.loc[~competency_dataset_check["exists"], "dataset_id"].tolist()
    raise AssertionError("Falta al menos un dataset esperado: " + ", ".join(missing))

print("Issue #16 PASS")
print("Saved:", dataset_check_output)

## 3. Issue #17 — Verificación de carga raw y trazabilidad

Verifica que las cuatro tablas raw existan y conserven conteos esperados de filas y columnas.

In [ ]:
raw_checks = []

for table_name, spec in RAW_TABLES.items():
    table_ref = f"{PROJECT_ID}.{RAW_DATASET}.{table_name}"
    try:
        table = client.get_table(table_ref)
        table_exists = True
        bq_rows = table.num_rows
        bq_columns = len(table.schema)
        schema_fields = "; ".join([f"{field.name}:{field.field_type}" for field in table.schema])
        error_message = None
    except Exception as exc:
        table_exists = False
        bq_rows = None
        bq_columns = None
        schema_fields = None
        error_message = str(exc)

    raw_checks.append({
        "project_id": PROJECT_ID,
        "dataset_id": RAW_DATASET,
        "source_file": spec["source_file"],
        "table_name": table_name,
        "full_table_id": table_ref,
        "table_exists": table_exists,
        "expected_rows": spec["expected_rows"],
        "bq_rows": bq_rows,
        "expected_columns": spec["expected_columns"],
        "bq_columns": bq_columns,
        "rowcount_match": table_exists and bq_rows == spec["expected_rows"],
        "column_count_match": table_exists and bq_columns == spec["expected_columns"],
        "schema_fields": schema_fields,
        "competency": "Bases de Datos - carga raw y trazabilidad",
        "checked_at_utc": RUN_UTC,
        "error_message": error_message,
    })

raw_competency_check = pd.DataFrame(raw_checks)

raw_check_output = f"{LOG_DIR}/ENARES_2024_STAGE2_competency_raw_load_check.csv"
raw_competency_check.to_csv(raw_check_output, index=False)

display(raw_competency_check[[
    "source_file",
    "table_name",
    "table_exists",
    "expected_rows",
    "bq_rows",
    "expected_columns",
    "bq_columns",
    "rowcount_match",
    "column_count_match",
]])

if not raw_competency_check["table_exists"].all():
    missing = raw_competency_check.loc[~raw_competency_check["table_exists"], "table_name"].tolist()
    raise AssertionError("Faltan tablas raw: " + ", ".join(missing))

if not raw_competency_check["rowcount_match"].all():
    failed = raw_competency_check.loc[~raw_competency_check["rowcount_match"], "table_name"].tolist()
    raise AssertionError("Falló validación de filas raw: " + ", ".join(failed))

if not raw_competency_check["column_count_match"].all():
    failed = raw_competency_check.loc[~raw_competency_check["column_count_match"], "table_name"].tolist()
    raise AssertionError("Falló validación de columnas raw: " + ", ".join(failed))

print("Issue #17 PASS")
print("Saved:", raw_check_output)

## 4. Issue #18 — Verificación de metadata y gobernanza

Verifica la existencia de las tablas de metadata SPSS generadas por el Notebook 03.

In [ ]:
metadata_checks = []

for table_name in METADATA_TABLES:
    table_ref = f"{PROJECT_ID}.{RAW_DATASET}.{table_name}"
    try:
        table = client.get_table(table_ref)
        exists = True
        row_count = table.num_rows
        column_count = len(table.schema)
        schema_fields = "; ".join([f"{field.name}:{field.field_type}" for field in table.schema])
        error_message = None
    except Exception as exc:
        exists = False
        row_count = None
        column_count = None
        schema_fields = None
        error_message = str(exc)

    metadata_checks.append({
        "project_id": PROJECT_ID,
        "dataset_id": RAW_DATASET,
        "metadata_table": table_name,
        "full_table_id": table_ref,
        "exists": exists,
        "row_count": row_count,
        "column_count": column_count,
        "schema_fields": schema_fields,
        "competency": "Metadata y gobernanza",
        "checked_at_utc": RUN_UTC,
        "error_message": error_message,
    })

metadata_competency_check = pd.DataFrame(metadata_checks)

metadata_check_output = f"{LOG_DIR}/ENARES_2024_STAGE2_competency_metadata_check.csv"
metadata_competency_check.to_csv(metadata_check_output, index=False)

display(metadata_competency_check[[
    "metadata_table",
    "exists",
    "row_count",
    "column_count",
]])

if not metadata_competency_check["exists"].all():
    missing = metadata_competency_check.loc[~metadata_competency_check["exists"], "metadata_table"].tolist()
    raise AssertionError("Faltan tablas de metadata requeridas: " + ", ".join(missing))

print("Issue #18 metadata tables PASS")
print("Saved:", metadata_check_output)

## 5. Issue #18 — Verificación específica de metadata PDF

Verifica que `metadata_crs04_source_files` incluya las columnas requeridas por el anexo para preservar metadata documental PDF.

Si los Drive IDs no están completos, el check deja evidencia de limitación; no inventa valores.

In [ ]:
# ============================================================
# 5. Issue #18 — Verificación específica de metadata PDF
# ============================================================

required_pdf_columns = [
    "questionnaire_pdf_file",
    "questionnaire_pdf_sha256",
    "questionnaire_pdf_drive_id",
    "variable_dictionary_pdf_file",
    "variable_dictionary_pdf_sha256",
    "variable_dictionary_pdf_drive_id",
    "pdf_dictionary_extracted_to_table",
    "pdf_dictionary_extraction_notes",
]

source_files_ref = f"{PROJECT_ID}.{RAW_DATASET}.metadata_crs04_source_files"

try:
    source_table = client.get_table(source_files_ref)
    source_schema = {field.name for field in source_table.schema}
    source_files_exists = True
    source_files_error = None
except Exception as exc:
    source_schema = set()
    source_files_exists = False
    source_files_error = str(exc)

pdf_schema_checks = [
    {
        "check_type": "required_column_present",
        "required_column": col,
        "present": col in source_schema,
        "competency": "Metadata documental PDF y gobernanza",
        "checked_at_utc": RUN_UTC,
    }
    for col in required_pdf_columns
]

pdf_schema_check = pd.DataFrame(pdf_schema_checks)

# Detectar nombre real de columna del archivo fuente
source_file_col = (
    "source_file"
    if "source_file" in source_schema
    else "sav_file"
    if "sav_file" in source_schema
    else None
)

if source_file_col is None:
    raise AssertionError(
        "metadata_crs04_source_files no tiene columna source_file ni sav_file."
    )

if "target_table" not in source_schema:
    raise AssertionError(
        "metadata_crs04_source_files no tiene columna target_table."
    )

if source_files_exists:
    pdf_content_sql = f"""
    SELECT
        {source_file_col} AS source_file,
        target_table,
        questionnaire_pdf_file,
        questionnaire_pdf_sha256,
        questionnaire_pdf_drive_id,
        variable_dictionary_pdf_file,
        variable_dictionary_pdf_sha256,
        variable_dictionary_pdf_drive_id,
        pdf_dictionary_extracted_to_table,
        pdf_dictionary_extraction_notes
    FROM `{source_files_ref}`
    ORDER BY target_table
    """

    pdf_content_check = client.query(pdf_content_sql).result().to_dataframe()

    pdf_content_check["questionnaire_pdf_complete"] = (
      pdf_content_check["questionnaire_pdf_file"].notna()
      & pdf_content_check["questionnaire_pdf_sha256"].notna()
    )

    pdf_content_check["dictionary_pdf_complete"] = (
        pdf_content_check["variable_dictionary_pdf_file"].notna()
        & pdf_content_check["variable_dictionary_pdf_sha256"].notna()
        & pdf_content_check["variable_dictionary_pdf_drive_id"].notna()
    )
    pdf_content_check["questionnaire_pdf_drive_id_documented"] = (
    pdf_content_check["questionnaire_pdf_drive_id"].notna()
    )


    pdf_content_check["questionnaire_pdf_governance_note"] = (
        pdf_content_check["questionnaire_pdf_drive_id"]
        .isna()
        .map({
            True: "Drive ID unavailable in Stage 01 manifest (documented limitation)",
            False: "Drive ID available"
        })
    )

    pdf_content_check["pdf_governance_complete"] = (
        pdf_content_check["questionnaire_pdf_complete"]
        & pdf_content_check["dictionary_pdf_complete"]
    )

else:
    pdf_content_check = pd.DataFrame([{
        "source_file": None,
        "target_table": None,
        "questionnaire_pdf_complete": False,
        "dictionary_pdf_complete": False,
        "pdf_governance_complete": False,
        "error_message": source_files_error,
    }])

pdf_schema_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_competency_pdf_metadata_schema_check.csv"
)
pdf_content_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_competency_pdf_metadata_check.csv"
)

pdf_schema_check.to_csv(pdf_schema_output, index=False)
pdf_content_check.to_csv(pdf_content_output, index=False)

display(pdf_schema_check)
display(pdf_content_check)

if not pdf_schema_check["present"].all():
    missing_cols = pdf_schema_check.loc[
        ~pdf_schema_check["present"], "required_column"
    ].tolist()

    raise AssertionError(
        "metadata_crs04_source_files no tiene todas las columnas PDF requeridas: "
        + ", ".join(missing_cols)
    )

if not pdf_content_check["pdf_governance_complete"].all():
    incomplete = pdf_content_check.loc[
        ~pdf_content_check["pdf_governance_complete"], "target_table"
    ].tolist()

    print("WARNING: PDF governance is incomplete for:", incomplete)
    print(
        "Complete ENARES_2024_STAGE2_pdf_registry_manual.csv "
        "si Drive IDs o clasificación oficial PDF están pendientes."
    )
else:
    print("Issue #18 PDF governance PASS")

print("Saved:", pdf_schema_output)
print("Saved:", pdf_content_output)

## 6. Observabilidad — tamaño de tablas raw

Genera evidencia técnica de filas, columnas y tamaño lógico de las tablas raw.

In [ ]:
observability_rows = []

for table_name in RAW_TABLES.keys():
    table_ref = f"{PROJECT_ID}.{RAW_DATASET}.{table_name}"
    table = client.get_table(table_ref)

    observability_rows.append({
        "project_id": PROJECT_ID,
        "dataset_id": RAW_DATASET,
        "table_name": table_name,
        "table_num_rows": table.num_rows,
        "table_num_bytes": table.num_bytes,
        "table_mb": round(table.num_bytes / 1024**2, 2),
        "num_columns": len(table.schema),
        "created": table.created.isoformat() if table.created else None,
        "modified": table.modified.isoformat() if table.modified else None,
        "checked_at_utc": RUN_UTC,
    })

ingestion_observability = pd.DataFrame(observability_rows)

observability_output = f"{LOG_DIR}/ENARES_2024_STAGE2_ingestion_observability.csv"
ingestion_observability.to_csv(observability_output, index=False)

display(ingestion_observability)
print("Saved:", observability_output)

## 7. Linaje — SHA-256 de archivos fuente

Usa `ENARES_2024_STAGE2_source_file_check.csv` de Notebook 02 para recalcular SHA-256 de los `.sav` origen.

Si existe un manifest de Stage 01 con hashes, intenta compararlos. Si no existe, documenta la limitación sin inventar evidencia.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

source_check_path = f"{LOG_DIR}/ENARES_2024_STAGE2_source_file_check.csv"

if not os.path.exists(source_check_path):
    raise FileNotFoundError("Missing ENARES_2024_STAGE2_source_file_check.csv. Run Notebook 02 first.")

source_check = pd.read_csv(source_check_path)

required_source_cols = ["module", "chapter", "source_file", "target_table", "source_path"]
missing_source_cols = [c for c in required_source_cols if c not in source_check.columns]
if missing_source_cols:
    raise ValueError("source_file_check CSV missing columns: " + ", ".join(missing_source_cols))

PRIMARY_DIR = f"{ROOT_DRIVE}/01BasesDatosPrimarias"
stage1_manifests = sorted(glob.glob(f"{PRIMARY_DIR}/ENARES_2024_STAGE1_manifest_*.json"))

stage1_sha = {}

if stage1_manifests:
    latest_manifest = stage1_manifests[-1]
    with open(latest_manifest, "r", encoding="utf-8") as f:
        entries = json.load(f)

    if isinstance(entries, list):
        iterable_entries = entries
    else:
        iterable_entries = entries.get("modules", [])

    for entry in iterable_entries:
        extracted_file = (
            entry.get("extracted_data_file")
            or entry.get("extracted_file")
            or entry.get("source_file")
        )
        extracted_sha = (
            entry.get("extracted_file_sha256")
            or entry.get("sha256")
            or entry.get("file_sha256")
        )
        if extracted_file and extracted_sha:
            stage1_sha[os.path.basename(extracted_file)] = extracted_sha

    manifest_used = latest_manifest
else:
    manifest_used = None

lineage_rows = []

for _, row in source_check.iterrows():
    source_path = row["source_path"]
    source_file = row["source_file"]

    if os.path.exists(source_path):
        stage2_sha = file_sha256(source_path)
        file_exists = True
        error_message = None
    else:
        stage2_sha = None
        file_exists = False
        error_message = "Source file not found"

    stage1_value = stage1_sha.get(source_file)

    lineage_rows.append({
        "module": row["module"],
        "chapter": row["chapter"],
        "source_file": source_file,
        "source_path": source_path,
        "target_table": row["target_table"],
        "source_file_exists": file_exists,
        "stage2_sha256": stage2_sha,
        "stage1_manifest_used": manifest_used,
        "stage1_sha256": stage1_value,
        "sha_match": bool(stage1_value is not None and stage2_sha == stage1_value),
        "comparison_status": (
            "match"
            if stage1_value is not None and stage2_sha == stage1_value
            else "stage1_hash_missing"
            if stage1_value is None
            else "mismatch"
        ),
        "checked_at_utc": RUN_UTC,
        "error_message": error_message,
    })

lineage = pd.DataFrame(lineage_rows)

lineage_output = f"{LOG_DIR}/ENARES_2024_STAGE2_lineage_source_files.csv"
lineage.to_csv(lineage_output, index=False)

display(lineage)

broken = lineage[(lineage["stage1_sha256"].notna()) & (~lineage["sha_match"])]
if not broken.empty:
    raise AssertionError(
        "Provenance rota: el .sav difiere del SHA-256 de Stage 01 en "
        + ", ".join(broken["source_file"].tolist())
    )

if lineage["stage1_sha256"].isna().any():
    print("WARNING: Some Stage 01 hashes were not found. Stage 2 SHA-256 was still recorded.")

print("Saved:", lineage_output)

## 8. Cierre parcial de competencias #16–#18

Consolida la evidencia generada por este notebook.

In [ ]:
required_competency_outputs = [
    "ENARES_2024_STAGE2_competency_stage_policy.csv",
    "ENARES_2024_STAGE2_competency_dataset_layers_check.csv",
    "ENARES_2024_STAGE2_competency_raw_load_check.csv",
    "ENARES_2024_STAGE2_competency_metadata_check.csv",
    "ENARES_2024_STAGE2_competency_pdf_metadata_schema_check.csv",
    "ENARES_2024_STAGE2_competency_pdf_metadata_check.csv",
    "ENARES_2024_STAGE2_ingestion_observability.csv",
    "ENARES_2024_STAGE2_lineage_source_files.csv",
]

closure_rows = []

for filename in required_competency_outputs:
    path = f"{LOG_DIR}/{filename}"
    closure_rows.append({
        "filename": filename,
        "path": path,
        "exists": os.path.exists(path),
        "checked_at_utc": RUN_UTC,
    })

competency_closure = pd.DataFrame(closure_rows)
competency_closure["stage2_competency_evidence_pass"] = competency_closure["exists"]

closure_output = f"{LOG_DIR}/ENARES_2024_STAGE2_competency_16_18_closure_check.csv"
competency_closure.to_csv(closure_output, index=False)

display(competency_closure)

if not competency_closure["exists"].all():
    missing = competency_closure.loc[~competency_closure["exists"], "filename"].tolist()
    raise FileNotFoundError("Faltan evidencias de competencias #16-#18: " + ", ".join(missing))

stage2_competency_16_18_pass = bool(competency_closure["exists"].all())

print("stage2_competency_16_18_pass:", stage2_competency_16_18_pass)
print("Saved:", closure_output)

## 9. Texto sugerido para reporte o sustentación

Stage 02 queda documentado como una fase de ingesta raw, preservación de metadata y validación estructural sin transformación analítica de microdatos. Este notebook consolida evidencias de competencias para arquitectura por capas, carga raw trazable, metadata y gobernanza, observabilidad y linaje. Todas las verificaciones respetan el límite metodológico de Stage 02: no merge, no recodificación, no indicadores, no modelos y no interpretación estadística.